In [1]:
from dotenv import load_dotenv
load_dotenv("../../../.env")

True

# 검증 01: checksum + splitter

API 호출 없이 로컬에서만 동작하는 함수들. 토큰 사용 없음.

테스트 PDF: `Kiwoom-ESG/pdf_files/2024_아모레퍼시픽.pdf` (약 114MB → split 테스트 가능)

In [4]:
import os, sys
import fitz
print(f"Python: {sys.version}")

from pathlib import Path

SAMPLE_PDF = str(Path("../../../../Kiwoom-ESG/pdf_files/2024_아모레퍼시픽.pdf").resolve())

assert os.path.isfile(SAMPLE_PDF), f"PDF 없음: {SAMPLE_PDF}"

file_size = os.path.getsize(SAMPLE_PDF)
doc = fitz.open(SAMPLE_PDF)
print(f"PDF: {os.path.basename(SAMPLE_PDF)}")
print(f"크기: {file_size:,} bytes ({file_size / 1024 / 1024:.1f} MB)")
print(f"페이지: {len(doc)}")
doc.close()

Python: 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
PDF: 2024_아모레퍼시픽.pdf
크기: 114,552,110 bytes (109.2 MB)
페이지: 125


---
## 1. FileChecksum (TypedDict)

In [2]:
from llm_request.checksum import FileChecksum

sample: FileChecksum = {
    "checksum": "sha256:abc123",
    "filename": "test.pdf",
    "size": 1024,
    "mime_type": "application/pdf",
}
print(f"타입: {type(sample)}")
print(f"키들: {list(sample.keys())}")
print(f"→ checksum, filename, size, mime_type 4개 필드")

타입: <class 'dict'>
키들: ['checksum', 'filename', 'size', 'mime_type']
→ checksum, filename, size, mime_type 4개 필드


---
## 2. file_checksum()

**입력**: 파일 경로 (str | Path)

**출력**: FileChecksum dict (checksum, filename, size, mime_type)

In [5]:
from llm_request.checksum import file_checksum
import time

# 2-1. 기본 호출 + 소요 시간 측정 (114MB 해싱)
start = time.time()
meta = file_checksum(SAMPLE_PDF)
elapsed = time.time() - start

print("[입력]")
print(f"  path: {SAMPLE_PDF}")
print()
print("[출력]")
print(f"  checksum : {meta['checksum']}")
print(f"  filename : {meta['filename']}")
print(f"  size     : {meta['size']:,} bytes ({meta['size'] / 1024 / 1024:.1f} MB)")
print(f"  mime_type: {meta['mime_type']}")
print(f"\n소요 시간: {elapsed:.3f}초 (114MB 전체 SHA-256 해싱)")

[입력]
  path: /home/rladustn1227/02.github/Kiwoom-ESG/pdf_files/2024_아모레퍼시픽.pdf

[출력]
  checksum : sha256:0a4061daf4881e149b8bf13dc92cbf2a0233733fbdf2d8f386acf8769dbe7d77
  filename : 2024_아모레퍼시픽.pdf
  size     : 114,552,110 bytes (109.2 MB)
  mime_type: application/pdf

소요 시간: 1.035초 (114MB 전체 SHA-256 해싱)


In [6]:
# 2-2. 같은 파일 → 같은 checksum
meta2 = file_checksum(SAMPLE_PDF)
print(f"동일 checksum: {meta['checksum'] == meta2['checksum']}")

동일 checksum: True


In [7]:
# 2-3. 다른 파일 → 다른 checksum
import tempfile
with tempfile.NamedTemporaryFile(suffix=".txt", delete=False) as f:
    f.write(b"different")
    diff_path = f.name

meta3 = file_checksum(diff_path)
print(f"다른 checksum: {meta['checksum'] != meta3['checksum']}")
os.unlink(diff_path)

다른 checksum: True


In [8]:
# 2-4. filename 오버라이드
meta4 = file_checksum(SAMPLE_PDF, filename="custom.pdf")
print(f"[입력] filename='custom.pdf'")
print(f"[출력] filename: {meta4['filename']}, checksum 동일: {meta4['checksum'] == meta['checksum']}")

[입력] filename='custom.pdf'
[출력] filename: custom.pdf, checksum 동일: True


In [9]:
# 2-5. mime_type 오버라이드
meta5 = file_checksum(SAMPLE_PDF, mime_type="custom/type")
print(f"[입력] mime_type='custom/type'")
print(f"[출력] mime_type: {meta5['mime_type']}")

[입력] mime_type='custom/type'
[출력] mime_type: custom/type


In [10]:
# 2-6. 존재하지 않는 파일 → FileNotFoundError
try:
    file_checksum("/nonexistent/path.pdf")
except FileNotFoundError as e:
    print(f"[입력] '/nonexistent/path.pdf'")
    print(f"[출력] FileNotFoundError: {e}")

[입력] '/nonexistent/path.pdf'
[출력] FileNotFoundError: [Errno 2] No such file or directory: '/nonexistent/path.pdf'


---
## 3. split_pdf()

**입력**: path (str), target_mb (int)

**출력**: 파트 파일 경로 리스트 (list[str])

페이지별 바이트 추정 + 누적 그룹핑 방식. 5% 안전 마진.
재분할 최대 깊이: 3 (무한 재귀 방지).


In [11]:
from llm_request.file_handler.splitter import split_pdf

# 3-1. 50MB 이하 target → 114MB 파일이니까 분할됨
print(f"[입력] path={os.path.basename(SAMPLE_PDF)}, target_mb=50")
print(f"  파일 크기: {file_size / 1024 / 1024:.1f} MB")
print(f"  바이트 추정 기반 누적 그룹핑 (5% 안전 마진)")
print()

start = time.time()
parts = split_pdf(SAMPLE_PDF, target_mb=50)
elapsed = time.time() - start

print(f"[출력] {len(parts)}개 파트 (소요: {elapsed:.2f}초)")
total_pages = 0
for i, p in enumerate(parts):
    sz = os.path.getsize(p)
    doc = fitz.open(p)
    pages = len(doc)
    total_pages += pages
    doc.close()
    over = " ⚠️ 50MB 초과!" if sz > 50 * 1024 * 1024 else ""
    print(f"  파트 {i+1}: {sz / 1024 / 1024:.1f} MB, {pages}페이지{over}")

In [12]:
# 3-2. 페이지 수 보존 확인
original_doc = fitz.open(SAMPLE_PDF)
original_pages = len(original_doc)
original_doc.close()

print(f"원본: {original_pages}페이지")
print(f"분할 후 합계: {total_pages}페이지")
print(f"보존: {original_pages == total_pages}")

원본: 125페이지
분할 후 합계: 125페이지
보존: True


In [13]:
# 3-3. 모든 파트가 유효한 PDF
for i, p in enumerate(parts):
    doc = fitz.open(p)
    print(f"  파트 {i+1}: {len(doc)}페이지, 유효 PDF: True")
    doc.close()

  파트 1: 42페이지, 유효 PDF: True
  파트 2: 21페이지, 유효 PDF: True
  파트 3: 11페이지, 유효 PDF: True
  파트 4: 10페이지, 유효 PDF: True
  파트 5: 41페이지, 유효 PDF: True


In [14]:
# 3-4. 모든 파트가 50MB 이하인지
all_under_50 = all(os.path.getsize(p) <= 50 * 1024 * 1024 for p in parts)
print(f"모든 파트 ≤50MB: {all_under_50}")
if not all_under_50:
    print("⚠️ 50MB 초과 파트 있음! Files API 업로드 거부될 수 있음")

모든 파트 ≤50MB: True


In [15]:
# 3-5. 임시 파일 정리
for p in parts:
    if p != SAMPLE_PDF:
        os.unlink(p)
        print(f"삭제: {os.path.basename(p)}")
print("정리 완료")

삭제: tmpikt3xf3q.pdf
삭제: tmpzebq0l4o.pdf
삭제: tmpikoi0dxn.pdf
삭제: tmpuckie42j.pdf
삭제: tmpstpbco42.pdf
정리 완료


In [16]:
# 3-6. 분할 불필요 (target보다 작은 경우)
# quickstart.pdf 같은 작은 파일로 테스트
small_pdf = str(Path("../quickstart.pdf").resolve()) if Path("../quickstart.pdf").exists() else None
if small_pdf:
    result = split_pdf(small_pdf, target_mb=50)
    print(f"[입력] {os.path.basename(small_pdf)} ({os.path.getsize(small_pdf)/1024:.0f} KB)")
    print(f"[출력] {len(result)}개, 원본 반환: {result[0] == small_pdf}")
else:
    print("quickstart.pdf 없어서 스킵")

[입력] quickstart.pdf (598 KB)
[출력] 1개, 원본 반환: True


In [17]:
# 3-7. 존재하지 않는 파일
try:
    split_pdf("/nonexistent.pdf")
except Exception as e:
    print(f"[입력] '/nonexistent.pdf'")
    print(f"[출력] {type(e).__name__}: {e}")

[입력] '/nonexistent.pdf'
[출력] FileNotFoundError: [Errno 2] No such file or directory: '/nonexistent.pdf'


---
## 정리

| 함수 | 입력 | 출력 | 셀 수 |
|------|------|------|-------|
| `FileChecksum` | — | TypedDict 구조 | 1 |
| `file_checksum()` | path, filename?, mime_type? | FileChecksum dict | 6 |
| `split_pdf()` | path, target_mb | list[str] 파트 경로 | 7 |